# AIC Z₁–Z₂ Covariance



**Figures:** aic_z1z2_covariance.svg

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import subprocess, time
import matplotlib, matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "svg.fonttype": "path", "mathtext.fontset": "cm",
    "font.family": "serif", "font.size": 14,
    "axes.labelsize": 16, "axes.titlesize": 15,
    "legend.fontsize": 11, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "lines.linewidth": 2.5, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE="#0072BD"; ORANGE="#D95319"; GREEN="#77AC30"; PURPLE="#7E2F8E"
OUT = "."

k = 1.0
n_eta = 40
nevents = 3_000_000
burn = 500_000
eta_vals = np.logspace(-1, 3, n_eta)

inp = " ".join(f"{v:.8f}" for v in eta_vals) + "\n"

print(f"k={k}, {n_eta} eta values, {nevents/1e6:.0f}M events each")
t0 = time.time()
proc = subprocess.run(
    [".", str(k), str(n_eta), str(nevents), str(burn)],
    input=inp, capture_output=True, text=True, timeout=120)
print(f"SSA done in {time.time()-t0:.1f}s")

# Parse
lines = proc.stdout.strip().split("\n")
mean_z1 = np.zeros(n_eta); mean_z2 = np.zeros(n_eta)
cov_z1z2 = np.zeros(n_eta)
var_z1 = np.zeros(n_eta); var_z2 = np.zeros(n_eta)
for i, line in enumerate(lines):
    vals = [float(x) for x in line.split()]
    mean_z1[i], mean_z2[i], cov_z1z2[i], var_z1[i], var_z2[i] = vals

print(f"Cov range: [{cov_z1z2.min():.4f}, {cov_z1z2.max():.4f}]")
print(f"E[Z1] range: [{mean_z1.min():.3f}, {mean_z1.max():.3f}]")
print(f"E[Z2] range: [{mean_z2.min():.3f}, {mean_z2.max():.3f}]")

# Correlation coefficient
corr = cov_z1z2 / np.sqrt(var_z1 * var_z2)

# ── Figure ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
fig.subplots_adjust(left=0.11, right=0.96, top=0.92, bottom=0.10,
                    hspace=0.15)
fig.suptitle(
    r"AIC controller species: stationary covariance of $Z_1, Z_2$ "
    r"vs. $\eta$  ($k\!=\!1$)",
    fontsize=14, fontweight="bold")

# Top: covariance and variances
ax1.plot(eta_vals, cov_z1z2, color=BLUE, lw=2.5,
         label=r"$\mathrm{Cov}(Z_1, Z_2)$")
ax1.plot(eta_vals, var_z1, color=ORANGE, lw=2.0, ls="--",
         label=r"$\mathrm{Var}(Z_1)$")
ax1.plot(eta_vals, var_z2, color=GREEN, lw=2.0, ls="--",
         label=r"$\mathrm{Var}(Z_2)$")
ax1.axhline(0, color="0.7", lw=0.8)
ax1.set_ylabel("Covariance / Variance")
ax1.set_xscale("log")
ax1.legend(loc="upper right", framealpha=0.95, edgecolor="0.75")
ax1.grid(True, alpha=0.15)

# Bottom: correlation coefficient
ax2.plot(eta_vals, corr, color=PURPLE, lw=2.5)
ax2.axhline(0, color="0.7", lw=0.8)
ax2.axhline(-1, color="0.85", lw=0.8, ls=":")
ax2.set_xlabel(r"Sequestration rate $\eta$")
ax2.set_ylabel(r"$\rho(Z_1, Z_2)$")
ax2.set_title(r"Correlation coefficient "
              r"$\rho = \mathrm{Cov}(Z_1,Z_2) / "
              r"\sqrt{\mathrm{Var}(Z_1)\,\mathrm{Var}(Z_2)}$",
              fontsize=13)
ax2.set_xscale("log")
ax2.set_ylim(-1.1, 0.3)
ax2.grid(True, alpha=0.15)

fig.savefig(f"{OUT}/aic_z1z2_covariance.svg", bbox_inches="tight")
fig.savefig(".", dpi=200,
            bbox_inches="tight")
plt.close()
print("Saved aic_z1z2_covariance.svg")